In [7]:
import pandas as pd
import re
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# --- Cleaning function ---
def clean_text(text):
    if not isinstance(text, str):
        return ""
    # remove urls
    text = re.sub(r"http\S+|www\S+", "", text)
    # remove emojis / symbols not in Devanagari range
    text = re.sub(r"[^\u0900-\u097F\s]", " ", text)
    # normalize multiple spaces
    text = re.sub(r"\s+", " ", text).strip()
    return text

# --- Load dataset (your file already has headers: text,sentiment) ---
df = pd.read_csv("balanced_ordered_dataset.csv")

# --- Clean text column ---
df["text"] = df["text"].apply(clean_text)

# --- Standardize labels ---
df["sentiment"] = df["sentiment"].str.lower().str.strip()

# --- Encode labels ---
encoder = LabelEncoder()
df["label_encoded"] = encoder.fit_transform(df["sentiment"])

print("Label classes:", encoder.classes_)  
# should be: ['negative', 'neutral', 'positive']

# --- First split: Train (70%) and Temp (30%) ---
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df["text"], df["label_encoded"],
    test_size=0.3,
    stratify=df["label_encoded"],
    random_state=42
)

# --- Second split: Validation (15%) and Test (15%) from Temp ---
val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels,
    test_size=0.5,   # Half of 30% = 15%
    stratify=temp_labels,
    random_state=42
)

# --- Save to CSV ---
train_df = pd.DataFrame({"text": train_texts, "label": train_labels})
val_df   = pd.DataFrame({"text": val_texts,   "label": val_labels})
test_df  = pd.DataFrame({"text": test_texts,  "label": test_labels})

train_df.to_csv("train_clean.csv", index=False)
val_df.to_csv("val_clean.csv", index=False)
test_df.to_csv("test_clean.csv", index=False)

print("✅ Clean train/val/test datasets saved as train_clean.csv, val_clean.csv & test_clean.csv")


Label classes: ['negative' 'neutral' 'positive']
✅ Clean train/val/test datasets saved as train_clean.csv, val_clean.csv & test_clean.csv
